# Exercise 01 — new activation functions, from scratch

## Your task

Subclass the engine's `Tensor` and add three activation functions that the engine does
not provide: **`sigmoid`**, **`swish`** and **`softplus`**. For each one you implement
both the forward value *and* the local `_backward` rule (the derivative), exactly like
the activations already in [`bert_cpu/engine.py`](../bert_cpu/engine.py).

Why a subclass? Because nothing is removed from the engine — you *extend* it. The
existing `Tensor` (with `+`, `*`, `@`, `tanh`, ...) is your foundation; you are adding
new leaves on top.

## The maths you need

With $\sigma$ the logistic sigmoid:

$$\sigma(x) = \frac{1}{1 + e^{-x}}
\qquad\qquad \sigma'(x) = \sigma(x)\,\bigl(1 - \sigma(x)\bigr)$$

$$\mathrm{swish}(x) = x\,\sigma(x)
\qquad\qquad \mathrm{swish}'(x) = \sigma(x) + x\,\sigma(x)\bigl(1 - \sigma(x)\bigr)$$

$$\mathrm{softplus}(x) = \ln\bigl(1 + e^{x}\bigr)
\qquad\qquad \mathrm{softplus}'(x) = \sigma(x)$$

## How to work

1. Run the setup cell below.
2. Fill in the three methods (remove each `raise NotImplementedError`) and run that cell.
3. Run the **grading** cell: it compares the gradients *your* `_backward` produced against
   finite differences, on three given equations of increasing complexity.
4. Run the **plot** cell to *see* your functions with the autograd derivative overlaid.

> **Tip:** an activation is just another op. Look at how `tanh` / `relu` are written in
> `bert_cpu/engine.py`: compute the forward array, build the output `Tensor` with `self`
> as its only child, then set `out._backward` to a closure that accumulates
> *(local derivative)* `* out.grad` into `self.grad`.

In [ ]:
# Run me first: make ``bert_cpu`` importable whether Jupyter was started in the
# project root or inside ``exercises/``, exactly like the scripts in this folder do.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "bert_cpu").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np

from bert_cpu.engine import Tensor
from exercises.grading import check_equation

print("ready — engine imported from", ROOT)

## Your turn — the three activations

Fill in the three methods below. Each one must:

1. compute the forward array with NumPy,
2. build the output `Tensor` with `self` as its only child (that is what wires it into the
   computational graph),
3. give the output a `_backward` closure that adds *(local derivative)* `* out.grad` to
   `self.grad`.

In [ ]:
class ExTensor(Tensor):
    """A ``Tensor`` extended with extra activation functions."""

    def sigmoid(self) -> Tensor:
        """Logistic sigmoid, 1 / (1 + e^{-x})."""
        # TODO: 
        raise NotImplementedError("implement ExTensor.sigmoid")

    def swish(self) -> Tensor:
        """Swish / SiLU, x * sigmoid(x)."""
        # TODO: 
        raise NotImplementedError("implement ExTensor.swish")

    def softplus(self) -> Tensor:
        """Softplus, ln(1 + e^{x}) (a smooth ReLU)."""
        # TODO: 
        raise NotImplementedError("implement ExTensor.softplus")

## GIVEN — the three equations

You do not edit the next two cells. Once your activations work, the checker computes the
gradient of each equation with `backward()` and verifies it against finite differences.
Each equation is a function of one or more leaf tensors.

Note on `g3`: `wᵀ @ x + b` is produced by the base engine, so it is a plain `Tensor` —
calling `ExTensor.softplus(z)` applies *your* method to it (a method is just a function of
a tensor). Tensors are column-oriented here, as everywhere in the library: `x` and `w` are
`(3, 1)`, so `wᵀ @ x` is the familiar tiny neuron of the README.

In [ ]:
EQUATIONS = [
    (
        "g1(x) = mean( sigmoid(x) )",
        lambda x: x.sigmoid().mean(),
        [("x", (4, 1))],
    ),
    (
        "g2(x) = sum( swish(x) + softplus(x) )",
        lambda x: (x.swish() + x.softplus()).sum(),
        [("x", (4, 1))],
    ),
    (
        "g3(x, w, b) = sum( softplus(wᵀ @ x + b) )   # a tiny neuron",
        lambda x, w, b: ExTensor.softplus(w.T @ x + b).sum(),
        [("x", (3, 1)), ("w", (3, 1)), ("b", (1, 1))],
    ),
]

## Grade yourself

`check_equation` lives in [`exercises/grading.py`](grading.py) — the harness every
exercise in this folder shares. It perturbs each leaf numerically and compares the result
with the gradient your `_backward` wrote into `.grad`. A PASS means your derivative is the
real derivative of your forward.

In [ ]:
all_ok = True
for name, fn, specs in EQUATIONS:
    try:
        worst = check_equation(name, fn, specs, tensor_cls=ExTensor)
    except NotImplementedError as exc:
        all_ok = False
        print(f"  {name}\n    -> SKIPPED ({exc}).\n")
        continue
    all_ok = all_ok and worst < 1e-5

print("Your hand-written backward rules match finite differences."
      if all_ok else "Some derivatives are missing or off; revisit the _backward closures.")

## See them (optional — needs matplotlib)

The derivative curve below is *not* hand-coded: for an elementwise function,
`y.sum().backward()` leaves `x.grad[i] = f'(x_i)`, so the dashed line literally draws the
gradient your `_backward` produced.

Install the plotting dependency with `pip install matplotlib` if the cell says it is
missing — the engine itself stays NumPy-only.

In [ ]:
def plot_activations() -> None:
    """Plot each activation and, overlaid, the derivative your autograd gives."""
    import matplotlib.pyplot as plt

    xs = np.linspace(-6.0, 6.0, 400)
    names = ["sigmoid", "swish", "softplus"]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, name in zip(axes, names):
        x = ExTensor(xs.copy())
        y = getattr(x, name)()       # forward (your implementation)
        y.sum().backward()           # backward -> x.grad holds f'(x)
        ax.plot(xs, y.data, label=f"{name}(x)")
        ax.plot(xs, x.grad, "--", label=f"{name}'(x)  [autograd]")
        ax.set_title(name)
        ax.axhline(0, color="gray", lw=0.5)
        ax.axvline(0, color="gray", lw=0.5)
        ax.legend()
    fig.tight_layout()
    plt.show()


try:
    plot_activations()
except ImportError:
    print("matplotlib is not installed — run `pip install matplotlib` to see the curves.")
except NotImplementedError as exc:
    print(f"Implement the activations first ({exc}).")

---

**Next:** [Exercise 02 — Rewrite the Stars](q02_rewrite_the_stars.ipynb). There you add no
new op at all: you only *compose* ops the engine already differentiates, so there is no
`_backward` left to write.